# Train Enhanced Super-Resolution Generative Adversarial Network (ESRGAN) <BR> on Images from DIV2K and Flickr2K Datasets

Version 1.3:
* Discriminator padding and kernel size was altered
* Generator upsampling was altered
* Input images normalization is optional, via configuration parameter

Source:

https://github.com/wonbeomjang/ESRGAN-pytorch

Adapted by:

Antonio Esteves @ UMinho, Aug 2024

***

TODO:

* Modify `'OUR_WANDB_PROJECT_ID'`
* Modify `'OUR_WANDB_ENTITY'`
* Modify `LOAD_TRAINED_GENERATOR`
* Modify `LOAD_TRAINED_DISCRIMINATOR`
* Modify `SKIP_TRAIN_MODEL`
* Modify `CONFIG_FILE` for stage 1 or stage 2.
* In `../config/esrgan_05_stage1.yaml` file, modify the hyperparameters `experiment_name`, `root_dir`, `data_path`.
* In `../config/esrgan_05_stage12.yaml` file, modify the hyperparameters `experiment_name`, `root_dir`, `data_path`.

In [ ]:
import os
from   glob                              import glob
from   random                            import random
from   PIL                               import Image
import numpy                             as     np
import yaml
import wandb
import time

import torch
import torch.nn                          as     nn
from   torch.utils.data                  import Dataset, DataLoader
from   torch.optim.adam                  import Adam
import torch.nn.functional               as     F
from   torchvision.utils                 import save_image
import torchvision.transforms.functional as     transF
from   torchvision.models.vgg            import vgg19
from   torchinfo                         import summary
import matplotlib.pyplot                 as     plt

## Read the Configuration

In [ ]:
LOAD_TRAINED_GENERATOR     = False
LOAD_TRAINED_DISCRIMINATOR = False
SKIP_TRAIN_MODEL           = False

CONFIG_FILE        = '../config/esrgan_05_stage1.yaml'

with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

config["models_path"]  = os.path.join( config["root_dir"],  config["models_dir"],  config["experiment_name"])
config["results_path"] = os.path.join( config["root_dir"],  config["results_dir"], config["experiment_name"])

In [ ]:
print('parameters:')
for key, value in config.items():
    print(f'\t{key}: {value}')

In [ ]:
# Setup device agnostic code

device  = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} for computing')

## Utility functions

In [ ]:
def denorm(x):
    '''
    Denormalizes the tensor 'x', by adding '1', dividing by '2',
    # and clipping the values to the range [0,1].
    '''
    out = (x + 1) / 2
    return out.clamp_(0, 1)


def make_folder(path):
    '''
    Creates the folder 'path' if it does not exist.
    '''
    if not os.path.exists(path,):
        os.makedirs(path)


def time_format(seconds: int) -> str:
    '''
    Converts a time in seconds to a formatted string in the form: 01D:12H:34m:56s.
    '''
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'

## Login into Weights and Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights and Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
class Datasets(Dataset):
    '''
    Create a custom dataset containing pairs:
    (high resolution image, low resolution image).
    '''
    def __init__(self, data_path, image_size, scale, normalize):
        self.image_size = image_size
        self.scale      = scale
        self.data_path  = data_path
        self.normalize  = normalize

        if not os.path.exists(data_path):
            raise Exception(f"[ERROR] dataset does not exist")

        self.image_file_name = sorted(os.listdir(os.path.join(data_path, 'hr')))

    def __getitem__(self, item):
        file_name           = self.image_file_name[item]
        high_resolution     = Image.open(
            os.path.join(self.data_path, 'hr', file_name)
        ).convert('RGB')
        low_resolution      = Image.open(
            os.path.join(self.data_path, 'lr', file_name)
        ).convert('RGB')

        if random() > 0.5:
            high_resolution = transF.vflip(high_resolution)
            low_resolution  = transF.vflip(low_resolution)

        if random() > 0.5:
            high_resolution = transF.hflip(high_resolution)
            low_resolution  = transF.hflip(low_resolution)

        high_resolution     = transF.to_tensor(high_resolution)
        low_resolution      = transF.to_tensor(low_resolution)

        if self.normalize is True:
            high_resolution = transF.normalize(high_resolution, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
            low_resolution  = transF.normalize(low_resolution,  (0.5, 0.5, 0.5), (0.5, 0.5, 0.5))

        images = {'lr': low_resolution, 'hr': high_resolution}

        return images

    def __len__(self):
        return len(self.image_file_name)

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb
)

## Create a Training Dataset for Super-Resolution

In [ ]:
def get_loader(data_path, image_size, scale, batch_size, sample_batch_size, normalize_images):
    '''
    Create a Dataset and then a DataLoader for training.
    '''
    train_dataset = Datasets(data_path, image_size, scale, normalize_images)
    train_loader  = torch.utils.data.DataLoader(
        dataset    = train_dataset,
        batch_size = batch_size,
        shuffle    = True,
    )
    return train_loader

In [ ]:
class UpsamplingBlock(nn.Module):
    '''
    Upsampling Block.
    '''
    def __init__(self, nf, scale_factor=4):
        super(UpsamplingBlock, self).__init__()

        self.scale_factor = scale_factor
        if (self.scale_factor == 4):
            self.upsampling1 = nn.Sequential(
                nn.Conv2d(nf, nf, (3, 3), (1, 1), (1, 1)),
                nn.LeakyReLU(0.2, True)
            )
            self.upsampling2 = nn.Sequential(
                nn.Conv2d(nf, nf, (3, 3), (1, 1), (1, 1)),
                nn.LeakyReLU(0.2, True)
            )
        else:
            print('[ERROR] Unsupported upsampling factor')
            pass

    def forward(self, x):
        if (self.scale_factor == 4):
            x = self.upsampling1(F.interpolate(x, scale_factor=2, mode="nearest"))
            x = self.upsampling2(F.interpolate(x, scale_factor=2, mode="nearest"))

        return x

In [ ]:
class ResidualDenseBlock(nn.Module):
    '''
    Dense Residual Block.
    It is composed of a sequence of 5 residual blocks.
    Each residual block 'i' (RBi) has a Conv2d, a LeakyReLU, and 
    a skip connections that are used to concatenates the dense block input 'x' 
    with the output from all the previous residual blocks in the sequence  
    RB1_out, ..., RBi_out. The input of each residual block is the result 
    of the concatenation.
    The output of the final residual block is multiplied by 'beta' and added 
    to  input 'x' to produce the Dense Block output.
    '''
    def __init__(self, nf, gc=32, beta=0.2):
        super(ResidualDenseBlock, self).__init__()

        self.layer1 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 0 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer2 = nn.Sequential(

            nn.Conv2d(
                in_channels  = nf + 1 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1, 
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer3 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 2 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer4 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 3 * gc,
                out_channels = gc,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )
        self.layer5 = nn.Sequential(
            nn.Conv2d(
                in_channels  = nf + 4 * gc,
                out_channels = nf,
                kernel_size  = 3,
                padding      = 1,
                bias         = True,
            ),
            nn.LeakyReLU()
        )

        self.beta = beta

    def forward(self, x):
        l1_out = self.layer1(x)
        l2_out = self.layer2(torch.cat((x, l1_out), 1))
        l3_out = self.layer3(torch.cat((x, l1_out, l2_out), 1))
        l4_out = self.layer4(torch.cat((x, l1_out, l2_out, l3_out), 1))
        l5_out = self.layer5(torch.cat((x, l1_out, l2_out, l3_out, l4_out), 1))
        return l5_out.mul(self.beta) + x

In [ ]:
class ResidualInResidualDenseBlock(nn.Module):
    '''
    Residual in Residual Dense Block.

    Sequence of 3 Dense Residual Blocks.
    The output of the final dense residual block is multiplied by 'beta' and added 
    to  input 'x' to produce the RRDB output.
    '''
    def __init__(self, nf, gc=32, beta=0.2):
        super(ResidualInResidualDenseBlock, self).__init__()

        self.layer1 = ResidualDenseBlock(nf, gc)
        self.layer2 = ResidualDenseBlock(nf, gc)
        self.layer3 = ResidualDenseBlock(nf, gc)
        self.beta   = beta

    def forward(self, x):
        out = self.layer1(x)
        out = self.layer2(out)
        out = self.layer3(out)
        return out.mul(self.beta) + x

In [ ]:
class Discriminator(nn.Module):
    '''
    ESRGAN discriminator model.

    The discriminator structure includes 4 times:
    * Conv2d(stride=1)->BN->LeakyRelu --> Conv2d(stride=2)->BN->LeakyRelu
    * a block with Conv2d->LeakyRelu->Conv2d
    * a final classifier with Linear->LeakyRelu->Linear.

    '''
    def __init__(self, in_channels=3, out_channels=64, image_size=128, conv_blocks=4):
        super(Discriminator, self).__init__()

        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.image_size   = image_size
        block             = []

        for _ in range(conv_blocks):
            block += [
                # Adds padding with values that are a reflection of the neigbors values 
                # of the input tensor border row/column.
                #nn.ReflectionPad2d(1),
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=1, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, True),
                # iteration 1 output: BS,64,H,W
                # iteration 2 output: BS,64*2,H/(2),W/(2)
                # iteration 3 output: BS,64*2*2,H/(2*2),W/(2*2)
                # iteration 4 output: BS,64*2*2*2,H/(2*2*2),W/(2*2*2) 
            ]
            in_channels = out_channels

            block += [
                #nn.ReflectionPad2d(1),
                nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_channels),
                nn.LeakyReLU(0.2, True)
                # iteration 1 output: BS,64,H/(2),W/(2)
                # iteration 2 output: BS,64*2,H/(2*2),W/(2*2)
                # iteration 3 output: BS,64*2*2,H/(2*2*2),W/(2*2*2)
                # iteration 4 output: BS,64*(2*2*2),H/(2*2*2*2),W/(2*2*2*2) = BS,512,8,8
            ]
            out_channels *= 2 # 64*(2*2*2*2) 

        out_channels //= 2             # 64*(2*2*2*2)/ 2 = 64*(2*2*2)
        in_channels    = out_channels  # 64*(2*2*2)

        block += [
            nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU(0.2),
            # Output shape: BS,64*(2*2*2),H/(2*2*2*2*2),W/(2*2*2*2*2) = BS,512,4,4
        ]

        nf  = self.out_channels*(2 ** (conv_blocks-1))   # 512
        H_W = self.image_size // (2 ** (conv_blocks+1))  # 4

        self.feature_extraction = nn.Sequential(*block)

        #self.avgpool = nn.AdaptiveAvgPool2d((512, 512)) 

        self.classification = nn.Sequential(
            nn.Linear(nf * H_W * H_W, 100),
            nn.LeakyReLU(0.2, True),
            nn.Linear(100, 1)
        )

    def forward(self, x):
        x = self.feature_extraction(x)
        # shape: BS,512,4,4
        x = x.view(x.size(0), -1)
        # shape : BS,8192
        x = self.classification(x)
        return x


In [ ]:
class Generator(nn.Module):
    '''
    ESRGAN generator model.

    The generator structure includes:
    * a block with Conv2d->ReLU
    * 'rrdb_blocks'=23 RRDB blocks
    * a block with Conv2d->ReLU
    * an upsample block
    * a block with Conv2d->ReLU
    '''
    def __init__(self, in_channels, out_channels, nf=64, gc=32, scale_factor=4, rrdb_blocks=23):
        super(Generator, self).__init__()

        self.conv1 = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(in_channels, nf, 3), nn.ReLU())

        basic_block_layer = []

        for _ in range(rrdb_blocks):
            basic_block_layer += [ResidualInResidualDenseBlock(nf, gc)]

        self.basic_block = nn.Sequential(*basic_block_layer)

        self.conv2    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, nf, 3), nn.ReLU())
        self.upsample = UpsamplingBlock(nf, scale_factor=scale_factor)
        self.conv3    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, nf, 3), nn.ReLU())
        self.conv4    = nn.Sequential(nn.ReflectionPad2d(1), nn.Conv2d(nf, out_channels, 3), nn.ReLU())

    def forward(self, x):
        x1 = self.conv1(x)
        x  = self.basic_block(x1)
        x  = self.conv2(x)
        x  = self.upsample(x + x1)
        x  = self.conv3(x)
        x  = self.conv4(x)
        return x

In [ ]:
from torchvision.models import VGG19_Weights

class PerceptualLoss(nn.Module):
    '''
    Calculates the generator perceptual loss using a VGG network.
    It is calculated as the L1 loss between the output of the 35-layers of VGG
    when using true high resolution images and super-resolution generated images.
    '''
    def __init__(self):
        super(PerceptualLoss, self).__init__()

        vgg          = vgg19(weights=VGG19_Weights.DEFAULT)

        # Select the first 35 layers of VGG structure
        loss_network = nn.Sequential(*list(vgg.features)[:35]).eval()

        # Define all these layers has no trainable.
        for param in loss_network.parameters():
            param.requires_grad = False
        self.loss_network = loss_network

        # Select L1 loss function
        self.l1_loss      = nn.L1Loss()

    def forward(self, high_resolution, fake_high_resolution):
        # Calculate the L1 loss between the output of the 35-layers of VGG when
        # using true high resolution images and super-resolution generated images.
        perception_loss = self.l1_loss(
            self.loss_network(high_resolution),
            self.loss_network(fake_high_resolution),
        )
        return perception_loss


In [ ]:
class Trainer:
    '''
    The ESRGAN training class.
    '''
    def __init__(self, config, data_loader, device):
        
        # Save the hyperparameters in class attributes .........................

        self.data_loader       = data_loader
        self.epochs            = config["epochs"]
        self.start_epoch       = config["start_epoch"]
        self.image_size        = config["image_size"]
        self.channels          = config["channels"]
        self.normalize_images  = config["normalize_images"]
        self.models_dir        = config["models_dir"]
        self.batch_size        = config["batch_size"]
        self.d_conv_dim        = config["d_conv_dim"]
        self.g_conv_dim        = config["g_conv_dim"]
        self.nf                = config["nf"]
        self.gc                = config["gc"]
        self.g_rrdb_blocks     = config["g_rrdb_blocks"]
        self.d_conv_blocks     = config["d_conv_blocks"]
        self.results_dir       = config["results_dir"]
        self.scale_factor      = config["scale_factor"]
        self.experiment_name   = config["experiment_name"]
        self.log_interval      = config["log_interval"]
        self.sampling_interval = config["sampling_interval"]
        self.checkp_interval   = config["checkp_interval"]
        self.device            = device

        # Training a PSNR-based generator (stage 1)
        if config["PSNR_or_GAN_generator"] == True: 
            self.lr                      = config["p_lr"]
            self.content_loss_factor     = config["p_content_loss_factor"]
            self.perceptual_loss_factor  = config["p_perceptual_loss_factor"]
            self.adversarial_loss_factor = config["p_adversarial_loss_factor"]
            self.decay_iter              = config["p_decay_iter"]
            self.gamma                   = config["p_gamma"]

        # Training a GAN-based generator (stage 2)
        else:
            self.lr                      = config["g_lr"]
            self.content_loss_factor     = config["g_content_loss_factor"]
            self.perceptual_loss_factor  = config["g_perceptual_loss_factor"]
            self.adversarial_loss_factor = config["g_adversarial_loss_factor"]
            self.decay_iter              = config["g_decay_iter"]
            self.gamma                   = config["g_gamma"]

        # Instantiate the discriminator and generator models ...................

        self.build_model()

        # Select the optimizers: Adam for both discriminator and generator .....

        self.optimizer_generator = Adam(
            self.generator.parameters(),
            lr           = self.lr,
            betas        = (config["beta1"], config["beta2"]),
            weight_decay = config["weight_decay"],
        )
        self.optimizer_discriminator = Adam(
            self.discriminator.parameters(),
            lr           = self.lr,
            betas        = (config["beta1"], config["beta2"]),
            weight_decay = config["weight_decay"],
        )

        # Select the learning rate schedulers:              ....................
        # MultiStepLR for both discriminator and generator  ....................

        self.lr_scheduler_generator = torch.optim.lr_scheduler.MultiStepLR(
            optimizer  = self.optimizer_generator,
            milestones = self.decay_iter,
            gamma      = self.gamma,
        )
        self.lr_scheduler_discriminator = torch.optim.lr_scheduler.MultiStepLR(
            optimizer  = self.optimizer_discriminator,
            milestones = self.decay_iter,
            gamma      = self.gamma,
        )


    def train(self, config, results):
        '''
        Training function for ESRGAN model.
        '''
        total_steps           = len(self.data_loader)

        # Select the loss functions ............................................

        adversarial_criterion = nn.BCEWithLogitsLoss().to(self.device)
        content_criterion     = nn.L1Loss().to(self.device)
        perception_criterion  = PerceptualLoss().to(self.device)

        # Put the models in training mode ......................................

        self.generator.train()
        self.discriminator.train()

        # ----------------------------------------------------------------------
        # Training epochs loop
        # ----------------------------------------------------------------------
        for epoch in range(self.start_epoch, self.start_epoch+self.epochs):

            # ------------------------------------------------------------------
            # Epoch iterations loop
            # ------------------------------------------------------------------

            ts  = time.time()

            for step, image in enumerate(self.data_loader):

                low_resolution  = image['lr'].to(self.device)
                high_resolution = image['hr'].to(self.device)

                real_labels     = torch.ones((high_resolution.size(0), 1)).to(self.device)
                fake_labels     = torch.zeros((high_resolution.size(0), 1)).to(self.device)

                #...............................................................
                # Train the generator
                #...............................................................

                self.optimizer_generator.zero_grad()
                fake_high_resolution = self.generator(low_resolution)

                score_real           = self.discriminator(high_resolution)
                score_fake           = self.discriminator(fake_high_resolution)
                difference_r_f       = score_real - score_fake.mean()
                difference_f_r       = score_fake - score_real.mean()

                adversarial_loss_r_f = adversarial_criterion(
                    difference_r_f,
                    fake_labels,
                )
                adversarial_loss_f_r = adversarial_criterion(
                    difference_f_r,
                    real_labels,
                )
                adversarial_loss    = (adversarial_loss_f_r + adversarial_loss_r_f) / 2

                perceptual_loss     = perception_criterion(
                    high_resolution,
                    fake_high_resolution,
                )
                content_loss        = content_criterion(
                    fake_high_resolution,
                    high_resolution,
                )

                generator_loss = adversarial_loss * self.adversarial_loss_factor + \
                                 perceptual_loss  * self.perceptual_loss_factor  + \
                                 content_loss     * self.content_loss_factor

                generator_loss.backward()
                self.optimizer_generator.step()

                #...............................................................
                # Train the discriminator
                #...............................................................

                self.optimizer_discriminator.zero_grad()

                score_real           = self.discriminator(high_resolution)
                score_fake           = self.discriminator(fake_high_resolution.detach())
                difference_r_f       = score_real - score_fake.mean()
                difference_f_r       = score_fake - score_real.mean()

                adversarial_loss_r_f = adversarial_criterion(difference_r_f, real_labels)
                adversarial_loss_f_r = adversarial_criterion(difference_f_r, fake_labels)
                discriminator_loss   = (adversarial_loss_f_r + adversarial_loss_r_f) / 2

                discriminator_loss.backward()
                self.optimizer_discriminator.step()

                # Read current learning rates
                d_lr = self.optimizer_discriminator.param_groups[0]["lr"]
                g_lr = self.optimizer_generator.param_groups[0]["lr"]

                # Save the results in a dictionary .............................

                results['d_loss'].append(discriminator_loss.item())
                results['g_adversarial_loss'].append(adversarial_loss.item())
                results['g_perceptual_loss'].append(perceptual_loss.item())
                results['g_content_loss'].append(content_loss.item())
                results['g_loss'].append(generator_loss.item())
                results['d_lr'].append(d_lr)
                results['g_lr'].append(g_lr)

                # Print progress information ...................................

                if step % self.log_interval == 0:

                    mean_d_loss             = np.mean(results["d_loss"][-self.log_interval:])
                    mean_g_adversarial_loss = np.mean(results["g_adversarial_loss"][-self.log_interval:])
                    mean_g_perceptual_loss  = np.mean(results["g_perceptual_loss"][-self.log_interval:])
                    mean_g_content_loss     = np.mean(results["g_content_loss"][-self.log_interval:])
                    mean_g_loss             = np.mean(results["g_loss"][-self.log_interval:])

                    print(f'[Epoch {epoch+1}/{self.start_epoch+self.epochs} Batch {step+1}/{total_steps}]', end=" ")
                    print(f'Dloss {mean_d_loss :.6f} Gloss {mean_g_loss :.6f}', end=" ")
                    print(f'GadvLoss {mean_g_adversarial_loss * self.adversarial_loss_factor:.6f}', end=" ")
                    print(f'GpercLoss {mean_g_perceptual_loss * self.perceptual_loss_factor:.6f}', end=" ")
                    print(f'GcontLoss {mean_g_content_loss * self.content_loss_factor:.6f}', end=" ")
                    print(f'D_lr {d_lr}  G_lr {g_lr}')

                    try:
                        # Log metrics to Weights and Biases ....................
                        wandb.log(
                            {
                            "d_loss":         mean_d_loss,
                            "g_adv_loss":     mean_g_adversarial_loss,
                            "g_percep_loss":  mean_g_perceptual_loss,
                            "g_content_loss": mean_g_content_loss,
                            "g_loss":         mean_g_loss,
                            "d_LR":           d_lr,
                            "g_LR":           g_lr,
                            }
                        )
                    except Exception as ex:
                        print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

            # End of an epoch ..................................................

            # Update learning rates at the end of configured epochs.

            self.lr_scheduler_generator.step()
            self.lr_scheduler_discriminator.step()

            te        = time.time()
            texec_sec = te - ts
            texec_str = time_format(texec_sec)
            print(f'Epoch training time: {texec_str}')
            results['epoch_training_time'].append(texec_sec)

            # Log epoch exection time to Weights and Biases ....................
            try:
                wandb.log(
                    {
                    "epoch_training_time_sec": texec_sec,
                    }
                )
            except Exception as ex:
                print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

            # Plot and save 8 LR images, 8 SR images, and 8 HR images ..........

            if ((epoch+1) % self.sampling_interval == 0) or ((epoch+1) == (self.start_epoch+self.epochs)):

                self.plot_LR_SR_HR_images(
                    8,
                    epoch,
                    low_resolution,
                    fake_high_resolution,
                    high_resolution,
                )

            # Save a model checkpoint ..........................................

            if ((epoch+1) % self.checkp_interval == 0) or ((epoch+1) == (self.start_epoch+self.epochs)):
                file_save_model = f'models/{self.experiment_name}/{self.experiment_name}_{str(epoch+1).zfill(3)}.pth'
                self.save_model_and_results(
                    results,
                    epoch+1,
                    config,
                    file_save_model,
                )


    def build_model(self):
        '''
        Instantiated the discriminator and generator models.
        It also print the model architectures.
        '''
        self.generator = Generator(
            in_channels    = self.channels, 
            out_channels   = self.channels, 
            nf             = self.nf,
            gc             = self.gc,
            scale_factor   = self.scale_factor,
            rrdb_blocks    = self.g_rrdb_blocks,
        ).to(self.device)

        self.discriminator = Discriminator(
            in_channels    = self.channels,
            out_channels   = self.d_conv_dim,
            image_size     = self.image_size,
            conv_blocks    = self.d_conv_blocks,
        ).to(self.device)

        # Print the model's architecture

        LRsize   = self.image_size // self.scale_factor
        aux_data = torch.randn(8, self.channels, LRsize, LRsize, device=device)
        summ = summary(
            self.generator,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(summ)

        aux_data = torch.randn(
            (
            8,               # any batch size is OK
            self.channels,
            self.image_size,
            self.image_size,
            )
        ).to(device)

        summ = summary(
            self.discriminator,
            input_data   = aux_data,
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(summ)


    def save_model_and_results(
            self,
            results,
            epoch,
            hyperparameters,
            file_name,
        ):
        '''
        Given the current Trainer object, this method saves to a file:
        (i)   the discriminator model weights,
        (ii)  the generator model weights,
        (iii) the discriminator optimizer state,
        (iv)  the generator optimizer state,
        (v)   the results saved during model training,
        (vi)  the actual training epoch number,
        (vii) the hyperparameters used to train the models.
        '''
        results_to_save = {
            'discriminator':   self.discriminator.state_dict(),
            'generator':       self.generator.state_dict(),
            'd_optimizer':     self.optimizer_discriminator.state_dict(),
            'g_optimizer':     self.optimizer_generator.state_dict(),
            'results':         results,
            'epoch':           epoch,
            'hyperparameters': hyperparameters,
        }

        torch.save(
            results_to_save,
            file_name,
        )


    def load_model_V2(self, file_name, loadD=True, loadG=True, loadOptimD=False, loadOptimG=False, device=device):
        '''
        Given the current Trainer object, this method loads from file 'file_name':
        (i)   the discriminator model weights   (if loadD=True),
        (ii)  the generator model weights       (if loadG=True),
        (iii) the discriminator optimizer state (if loadD=True and loadOptimD=True),
        (iv)  the generator optimizer state     (if loadG=True and loadOptimG=True),
        (v)   the results saved during model training,
        (vi)  the training epoch number when the checkpoint was saved,
        (vii) the hyperparameters used to train the models.
        and put the models on 'device'.

        Returns the loaded results, the loaded epoch number, and the loaded hyperparameters.
        '''
        results_loaded = torch.load(file_name)

        if loadD == True:
            self.discriminator.load_state_dict(results_loaded['discriminator'])
            if loadOptimD==True:
                self.optimizer_discriminator.load_state_dict(results_loaded['d_optimizer'])
        self.discriminator.to(device)

        if loadG == True:
            self.generator.load_state_dict(results_loaded['generator'])
            if loadOptimG==True:
                self.optimizer_generator.load_state_dict(results_loaded['g_optimizer'])
        self.generator.to(device)

        self.start_epoch = results_loaded['epoch']

        # Returns the saved results and the saved hyperparameters
        return results_loaded['results'], \
                results_loaded['epoch'], \
                results_loaded['hyperparameters']


    def plot_LR_SR_HR_images(self, num_images, epoch, LR_image, SR_image, HR_image):
        '''
        PLot 'num_images' images of low resolution in the first row,
        the corresponding generated super resolution images in second row,
        and the corresponding high resolution images in third row,
        '''
        assert self.batch_size >= num_images, \
            f'Number of images must be less or equal to batch size={self.batch_size}'

        lr_cpu    = LR_image.cpu().permute(0,2,3,1) # BS,H,W,C
        sr_cpu    = SR_image.cpu().permute(0,2,3,1) # BS,H,W,C
        hr_cpu    = HR_image.cpu().permute(0,2,3,1) # BS,H,W,C

        if(self.batch_size > num_images):
            lr_cpu = lr_cpu[:num_images] # num_images,H,W,C
            sr_cpu = sr_cpu[:num_images] # num_images,H,W,C
            hr_cpu = hr_cpu[:num_images] # num_images,H,W,C

        gridLR = lr_cpu[0]
        for i in range(1, num_images):
            gridLR = torch.cat((gridLR, lr_cpu[i]), dim=1)

        gridSR = sr_cpu[0]
        for i in range(1, num_images):
            gridSR = torch.cat((gridSR, sr_cpu[i]), dim=1)

        gridHR = hr_cpu[0]
        for i in range(1, num_images):
            gridHR = torch.cat((gridHR, hr_cpu[i]), dim=1)

        if self.normalize_images == True:
            gridLR = denorm(gridLR)
            gridSR = denorm(gridSR)
            gridHR = denorm(gridHR)

        gridLR = gridLR.detach().numpy()
        gridSR = gridSR.detach().numpy()
        gridHR = gridHR.detach().numpy()

        # Display the grid of images

        img_dim = 4
        _, ax   = plt.subplots(3, 1, figsize=(num_images*img_dim, 3*img_dim))

        ax[0].imshow(gridLR) 
        ax[0].set_title(f'low resolution images at epoch {epoch+1}')
        ax[1].imshow(gridSR) 
        ax[1].set_title(f'generated super-resolution images at epoch {epoch+1}')
        ax[2].imshow(gridHR)
        ax[2].set_title(f'high resolution images at epoch {epoch+1}')

        file_png = f'results/{self.experiment_name}/{self.experiment_name}_LR_SR_HR_{str(epoch+1).zfill(3)}.png'
        plt.savefig(file_png, format='png')
        plt.show()
        plt.close() 


    def plot_single_LR_SR_HR_image(self, LR_image, SR_image, HR_image, epoch):
        '''
        Plot and save a single LR image, an SR image, amd a HR image.
        '''

        assert len(LR_image.shape) == 3 and len(SR_image.shape) == 3 and len(HR_image.shape) == 3, \
            f'The provided images must have 3 dimensions.'

        lr_cpu    = LR_image.cpu().permute(1,2,0).numpy()
        sr_cpu    = SR_image.cpu().permute(1,2,0).numpy()
        hr_cpu    = HR_image.cpu().permute(1,2,0).numpy()

        if self.normalize_images == True:
            lr_cpu    = denorm(lr_cpu)
            sr_cpu    = denorm(sr_cpu)
            hr_cpu    = denorm(hr_cpu)

        _, ax     = plt.subplots(1, 3, figsize=(3*5,1*5))
        plt.suptitle(
            f'Low-resolution  | generated super-resolution | high-resolution at epoch {epoch+1}',
            fontsize   = 15,
            fontweight = 'bold',
        )

        ax[0].imshow(lr_cpu) 
        ax[0].set_title('low resolution')
        ax[1].imshow(sr_cpu) 
        ax[1].set_title('generated super-resolution')
        ax[2].imshow(hr_cpu)
        ax[2].set_title('high resolution')

        file_png = f'results/{config["experiment_name"]}/{config["experiment_name"]}_LR_SR_HR_{str(epoch+1).zfill(3)}.png'
        plt.savefig(file_png, format='png')
        plt.show()
        plt.close() 


In [ ]:
# Create an empty dictionary to store the training results

results = {
    'd_loss':              [],
    'g_adversarial_loss':  [],
    'g_perceptual_loss':   [],
    'g_content_loss':      [],
    'g_loss':              [],
    'd_lr':                [],
    'g_lr':                [],
    'epoch_training_time': [],
}

In [ ]:
# Make directories if they do not exist

make_folder(config["models_path"])
make_folder(config["results_path"])

print(f"Start training of ESRGAN")

# Create the DataLoader

data_loader = get_loader(
    config["data_path"],
    config["image_size"],
    config["scale_factor"],
    config["batch_size"],
    config["sample_batch_size"],
    config["normalize_images"],
)

# Create the 'trainer' object used for training

trainer  = Trainer(config, data_loader, device)

In [ ]:
# =========================================================================
# Train the model from the beginning
# =========================================================================

if LOAD_TRAINED_GENERATOR == False and LOAD_TRAINED_DISCRIMINATOR == False and SKIP_TRAIN_MODEL == False:

    config["start_epoch"] = 0
    trainer.train(config, results)

# =========================================================================
# Load the saved models
# =========================================================================

elif LOAD_TRAINED_GENERATOR == True or LOAD_TRAINED_DISCRIMINATOR == True:

    file_save_model = f'models/{config["experiment_name"]}.pth'
    results, config["start_epoch"], _ = trainer.load_model_V2(
        file_save_model,
        loadD      = LOAD_TRAINED_DISCRIMINATOR,
        loadG      = LOAD_TRAINED_GENERATOR,
        loadOptimD = False,
        loadOptimG = False,
        device     = device,
    )

    #..................................................................
    # Comment these 2 lines when continuing to train a saved checkpoint
    #..................................................................
    trainer.start_epoch   = 0
    config["start_epoch"] = 0

    # ---------------------------------------------------------------------
    # Continue training of the loaded models
    # ---------------------------------------------------------------------

    if SKIP_TRAIN_MODEL == False:

        trainer.train(config, results)

In [ ]:
print(trainer.start_epoch)

## Export results to a CSV file

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    list(
        zip(
            results['d_loss'],
            results['g_adversarial_loss'],
            results['g_perceptual_loss'],
            results['g_content_loss'],
            results['g_loss'],
            results['d_lr'],
            results['g_lr'],
            results['epoch_training_time'],
        )
    ),
    columns = 
        [
            'd_loss',
            'g_adversarial_loss',
            'g_perceptual_loss',
            'g_content_loss',
            'g_loss',
            'd_lr',
            'g_lr',
            'epoch_training_time'
        ]
)
results_df.head()

file_name = f'results/{config["experiment_name"]}_results.csv'
results_df.to_csv(file_name)

In [ ]:
# Mark the Weights and Bias run as finished
wandb.finish()